# Slot 6 · Break it: is it biology, or something that looks like it?

**AI Days 2026 · Teaching Machines to Read DNA** — Technion

Two models now separate bound from unbound AGO2 sites:

| model | what it is | held-out AUROC |
|---|---|---|
| your CNN (Slot 2) | 10 motif detectors, trained on 92k sequences | ~0.86 |
| Mistral-DNA + probe (Slot 5) | genome-pretrained language model, 12k sequences | ~0.78 |

A score is not an explanation. Before we believe either model reads *binding sites*, we try three
cheap ways it could be cheating — each one a model that **should lose**:

1. **Composition:** just *how many* G/C and U letters a sequence has (2 numbers).
2. **Word counts:** how often each 5-letter word appears — no order, no position.
3. **The shuffle test:** keep a sequence's letters (and letter pairs) but destroy its order.
   A model that reads motifs should be fooled; a model that reads composition won't notice.

| cell | what you do |
|---|---|
| `#@title ... run this cell, nothing to change` | just run it |
| `# TODO-N` with `# === Your code here ===` | write it (yourself or with an assistant), then **check** it |

In [ ]:
#@title Setup - run this cell, nothing to change { display-mode: "form" }
import os, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from sklearn.metrics import roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - Runtime > Change runtime type > T4 GPU")

# Where the workshop files live. Either one direct-download link per file (FILE_URLS, e.g. SharePoint
# "anyone with the link" + download=1), or one folder URL (DATA_URL, e.g. a Hugging Face dataset repo).
FILE_URLS = {}
DATA_URL = os.environ.get("AIDAYS_DATA_URL", "https://huggingface.co/datasets/yup8/ai-days-2026-ago2/resolve/main")
DATA_DIR = Path(os.environ.get("AIDAYS_DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def fetch(name):
    """Return the local path of a workshop file, downloading it first if needed."""
    path = DATA_DIR / name
    if not path.exists():
        url = FILE_URLS.get(name) or (f"{DATA_URL.rstrip('/')}/{name}" if DATA_URL else "")
        if not url:
            raise FileNotFoundError(f"{path} not found and no download link is set for it")
        t = time.time()
        urllib.request.urlretrieve(url, path)
        print(f"downloaded {name} ({path.stat().st_size / 1e6:.1f} MB) in {time.time() - t:.1f}s")
    return path


def download_outputs(*names):
    """End of a notebook: save your results to your laptop so the next notebook can use them."""
    for n in names:
        p = DATA_DIR / n
        if not p.exists():
            print(f"{n}: not found (did the cell that makes it run?)")
        elif IN_COLAB:
            _colab_files.download(str(p))
        else:
            print(f"{n}: saved at {p}")


def upload_yours():
    """Start of a notebook: upload the files you downloaded at the end of an earlier notebook."""
    if not IN_COLAB:
        print(f"not in Colab: copy your files into {DATA_DIR}")
        return
    import re
    for n, b in _colab_files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", n)   # "my_cnn (1).pt" -> "my_cnn.pt" (browser renames)
        (DATA_DIR / clean).write_bytes(b)
        print("uploaded", n, "->", clean)

In [ ]:
#@title Data helpers - run this cell, nothing to change { display-mode: "form" }
MAX_LEN  = 501          # fixed global length (pad / truncate)
ALPHABET = "ACGU"       # order = the 4 rows of the one-hot matrix
FILES = {split: (f"CLIPSEQ_AGO2.{split}.positives.fa", f"CLIPSEQ_AGO2.{split}.negatives.fa")
         for split in ("train", "ls")}


def read_fasta(path, label):
    """Return (sequences, labels) from a FASTA file, all with the same label.

    The files mix the DNA letter T and the RNA letter U for the same base,
    so we normalise everything to U.
    """
    seqs, labs = [], []
    with Path(path).open() as fh:
        for line in fh:
            if line.startswith(">"):
                continue
            seqs.append(line.strip().upper().replace("T", "U"))
            labs.append(label)
    return seqs, labs


def read_split(split):
    """'train' or 'ls' (the held-out validation set) -> (sequences, labels as a numpy array)."""
    pos, neg = FILES[split]
    p, lp = read_fasta(fetch(pos), 1)
    n, ln = read_fasta(fetch(neg), 0)
    return p + n, np.array(lp + ln)


_LUT = np.full(256, -1, dtype=np.int64)
for _i, _b in enumerate(ALPHABET):
    _LUT[ord(_b)] = _i


def encode_batch(seqs, max_len=MAX_LEN):
    """List of RNA strings -> one-hot FloatTensor (B, 1, 4, max_len).

    Pads/truncates to max_len; padding and unknown bases stay uniform 0.25.
    """
    X = np.full((len(seqs), 1, 4, max_len), 0.25, dtype=np.float32)
    for b, s in enumerate(seqs):
        idx = _LUT[np.frombuffer(s[:max_len].encode(), dtype=np.uint8)]
        pos = np.nonzero(idx >= 0)[0]
        X[b, 0, :, pos] = 0.0
        X[b, 0, idx[pos], pos] = 1.0
    return torch.from_numpy(X)

In [ ]:
#@title Foundation-model helpers - run this cell, nothing to change { display-mode: "form" }
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import transformers
transformers.logging.set_verbosity_error()

FM_REPO = "RaphaelMourad/Mistral-DNA-v1-17M-hg38"            # 16M parameters, pretrained on the human genome
NT_REPO = "InstaDeepAI/nucleotide-transformer-500m-human-ref"  # 486M parameters


def to_dna(seq):
    """The DNA language models read T, not U."""
    return seq.replace("U", "T")


def workshop_data(n_train=12000, seed=0):
    """Train: a fixed random 12k subset of the training file (the same in Slots 5-6).
    Test: the held-out 'ls' set - the same test set the Slot 2 CNN was measured on."""
    seqs, y = read_split("train")
    tr = np.random.default_rng(seed).permutation(len(seqs))[:n_train]
    test_seqs, test_y = read_split("ls")
    return [seqs[i] for i in tr], y[tr], test_seqs, test_y


def load_fm(repo=FM_REPO):
    """Download (first time only) and load a DNA language model + its tokenizer."""
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
    model = AutoModel.from_pretrained(repo, trust_remote_code=True).to(DEVICE).eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token or tok.unk_token
    return tok, model

In [ ]:
#@title Embedding + probe helpers (written in Slot 5) - run this cell, nothing to change { display-mode: "form" }
@torch.no_grad()
def embed(tok, model, seqs, batch=128):
    """List of DNA strings -> (N, hidden) numpy array: the mean of the last layer over real tokens."""
    model.eval()
    out = []
    for i in range(0, len(seqs), batch):
        enc = tok(seqs[i:i + batch], return_tensors="pt", padding=True, truncation=True, max_length=512)
        ids, mask = enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            h = model(input_ids=ids, attention_mask=mask, output_hidden_states=True).hidden_states[-1].float()
        m = mask.unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out)


def probe_auc(E_train, y_train, E_test, y_test, C=0.1):
    """Fit a linear probe (logistic regression) on the training vectors; AUROC on the test vectors."""
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=C))
    probe.fit(E_train, y_train)
    return roc_auc_score(y_test, probe.predict_proba(E_test)[:, 1]), probe

In [ ]:
#@title The CNN from Slot 2 - run this cell, nothing to change { display-mode: "form" }
class GlobalCNN(nn.Module):
    """The Slot 2 network: one-hot (N,1,4,501) -> one binding score (logit) per sequence."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 10, (4, 10)), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Conv2d(10, 20, (1, 5)), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Flatten(),
            nn.Linear(20 * 53, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


@torch.no_grad()
def score_sequences(model, seqs, batch=1024):
    """The model's binding probability for each sequence (list of RNA strings)."""
    model.eval()
    out = []
    for i in range(0, len(seqs), batch):
        X = encode_batch(seqs[i:i + batch])
        out.append(torch.sigmoid(model(X.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


def load_cnn(mine="my_cnn.pt", ours="cnn.pt"):
    """Your CNN from Slot 2 if you uploaded it and it loads; otherwise ours."""
    model = GlobalCNN().to(DEVICE)
    if (DATA_DIR / mine).exists():
        try:
            model.load_state_dict(torch.load(DATA_DIR / mine, map_location=DEVICE))
            print("loaded YOUR CNN from Slot 2")
            return model.eval()
        except Exception as e:
            print(f"your {mine} doesn't match the Slot 2 architecture ({type(e).__name__}) - using ours")
    model.load_state_dict(torch.load(fetch(ours), map_location=DEVICE))
    print("loaded the reference CNN")
    return model.eval()

In [ ]:
#@title Shortcut toolkit - run this cell, nothing to change { display-mode: "form" }
from collections import defaultdict


def gc_u(seqs):
    """(N, 2): the fraction of G+C letters and the fraction of U (or T) letters in each sequence."""
    out = np.zeros((len(seqs), 2))
    for i, s in enumerate(seqs):
        n = max(len(s), 1)
        out[i] = ((s.count("G") + s.count("C")) / n, (s.count("U") + s.count("T")) / n)
    return out


_K = {b: i for i, b in enumerate("ACGT")}
_K["U"] = 3


def kmer_counts(seqs, k=5):
    """(N, 4^k): how often each k-letter word appears in each sequence (as a fraction)."""
    M = np.zeros((len(seqs), 4 ** k), dtype=np.float32)
    for r, s in enumerate(seqs):
        code, valid = 0, 0
        for j, ch in enumerate(s):
            v = _K.get(ch, -1)
            if v < 0:
                valid = 0
                continue
            code = (code * 4 + v) % (4 ** k)
            valid += 1
            if valid >= k:
                M[r, code] += 1
        M[r] /= max(M[r].sum(), 1)
    return M


def dinuc_shuffle(seq, rng):
    """Shuffle a sequence but keep every letter-pair count (Altschul-Erickson / Eulerian walk).

    The result has exactly the same letters, the same GC, the same letter pairs - but any longer
    pattern (a motif) is destroyed.
    """
    if len(seq) < 3:
        return seq
    edges = defaultdict(list)
    for a, b in zip(seq[:-1], seq[1:]):
        edges[a].append(b)
    last = seq[-1]
    while True:  # random spanning tree of "last exits" pointing towards the final letter
        last_edge = {v: edges[v][rng.integers(len(edges[v]))] for v in edges if v != last}
        ok = True
        for v in last_edge:
            seen, u = set(), v
            while u != last:
                if u in seen or u not in last_edge:
                    ok = False
                    break
                seen.add(u)
                u = last_edge[u]
            if not ok:
                break
        if ok:
            break
    order = {}
    for v, lst in edges.items():
        lst = list(lst)
        if v in last_edge:
            lst.remove(last_edge[v])
            rng.shuffle(lst)
            lst.append(last_edge[v])
        else:
            rng.shuffle(lst)
        order[v] = lst
    out, u, ptr = [seq[0]], seq[0], defaultdict(int)
    for _ in range(len(seq) - 1):
        nxt = order[u][ptr[u]]
        ptr[u] += 1
        out.append(nxt)
        u = nxt
    return "".join(out)

In [ ]:
#@title Rejoin here: the CNN (Slot 2) and the vectors (Slot 5), yours or ours - run this cell { display-mode: "form" }
USE_MY_FILES = False   # True = upload my_cnn.pt and/or my_emb_mistral17m.npz from Slots 2 and 5
if USE_MY_FILES:
    upload_yours()
S_train, y_train, S_test, y_test = workshop_data()
emb = DATA_DIR / "my_emb_mistral17m.npz"
d = np.load(emb if emb.exists() else fetch("emb_mistral17m.npz"))
print("vectors:", "YOURS from Slot 5" if emb.exists() else "the reference ones")
E_train, E_test = d["train"], d["test"]
cnn = load_cnn()
tok, fm = load_fm(FM_REPO)

table = {}
table["CNN (Slot 2)"] = roc_auc_score(y_test, score_sequences(cnn, S_test))
table["Mistral-DNA + probe (Slot 5)"], probe = probe_auc(E_train, y_train, E_test, y_test)
for k, v in table.items():
    print(f"{k:32s} test AUROC {v:.3f}")

## 1. Shortcut #1: composition

Look first: are bound and unbound sequences made of different letters?

In [ ]:
#@title Plot GC and U content - run this cell { display-mode: "form" }
import matplotlib.pyplot as plt
C = gc_u(S_train)
fig, ax = plt.subplots(1, 2, figsize=(8, 2.8))
for j, name in enumerate(("G+C fraction", "U fraction")):
    for lab, c, lbl in ((0, "tab:grey", "unbound"), (1, "tab:red", "bound")):
        ax[j].hist(C[y_train == lab, j], bins=40, alpha=.55, color=c, label=lbl, density=True)
    ax[j].set_xlabel(name); ax[j].legend()
plt.tight_layout(); plt.show()

### TODO-1 · A two-number model

Fit a logistic regression on just these **two numbers** per sequence and report its AUROC on the
test set. The tools are already here:

- `gc_u(seqs)` → an array of shape `(len(seqs), 2)`: G+C fraction and U fraction per sequence;
- `probe_auc(F_train, y_train, F_test, y_test)` → **returns `(auc, probe)`**; it works on any
  feature matrix, not only embeddings.

**Check:** `auc_gc` ≈ **0.66**.

In [ ]:
# TODO-1: AUROC of a model that only sees G+C and U content
# === Your code here ===
...
# === End of your code ===
table["G+C and U content only (2 numbers)"] = auc_gc
print(f"2 numbers: {auc_gc:.3f}")

## 2. Shortcut #2: counting words

`kmer_counts(seqs, k=5)` turns each sequence into 1,024 numbers: how often each 5-letter word
occurs. A linear model on those counts knows *which* words appear — but nothing about where or
in what order.

**Tools:** `kmer_counts(seqs, k=5)` → array `(len(seqs), 1024)`, then `probe_auc` as above.
**Check:** `auc_k5` ≈ **0.74**. (Counting 12,000 sequences takes ~5 s.)

In [ ]:
# TODO-2: AUROC of a linear model on 5-letter word counts (use kmer_counts + probe_auc)
# === Your code here ===
...
# === End of your code ===
table["5-letter word counts (1,024 numbers)"] = auc_k5
print(f"word counts: {auc_k5:.3f}")

## 3. The shuffle test

Take every **bound** test sequence and shuffle it with `dinuc_shuffle`: same letters, same
letter pairs, same GC — but motifs destroyed. Then ask each model: *can you still tell these
fake "bound" sequences from the real unbound ones?*

* AUROC stays **high** → the model is reading composition.
* AUROC drops towards **0.5** → the model was reading the order of letters: motifs.

In [ ]:
#@title Make the shuffled sequences - run this cell { display-mode: "form" }
rng = np.random.default_rng(0)
bound = [s for s, y in zip(S_test, y_test) if y == 1]
unbound = [s for s, y in zip(S_test, y_test) if y == 0]
shuffled = [dinuc_shuffle(s, rng) for s in bound]
print("original:", bound[0][:60])
print("shuffled:", shuffled[0][:60])
print("same G+C and U?", np.allclose(gc_u(bound), gc_u(shuffled)))
y_shuf = np.array([1] * len(shuffled) + [0] * len(unbound))

### TODO-3 · Run the shuffle test on both models

Compute two AUROCs, both against `y_shuf` (1 for the shuffled bound sequences, 0 for the real
unbound ones), for the list `shuffled + unbound`:

* **CNN** → `auc_shuf_cnn`: `score_sequences(cnn, seqs)` gives one probability per sequence.
* **Language model** → `auc_shuf_fm`: `embed(tok, fm, dna_seqs)` gives the vectors (the model reads
  T, not U: convert each sequence with `to_dna` first), then `probe.predict_proba(vectors)[:, 1]`.

**Gotchas to tell your assistant:** use the `probe` that was already fitted on the real training
vectors; do **not** fit a new one on shuffled data (that would answer a different question).

**Check:** CNN ≈ **0.66**, language model ≈ **0.77**.

In [ ]:
# TODO-3: shuffle-test AUROC for the CNN and for Mistral-DNA + probe
# === Your code here ===
...
# === End of your code ===
print(f"CNN         : real {table['CNN (Slot 2)']:.3f}  ->  shuffled {auc_shuf_cnn:.3f}")
print(f"Mistral-DNA : real {table['Mistral-DNA + probe (Slot 5)']:.3f}  ->  shuffled {auc_shuf_fm:.3f}")

In [ ]:
#@title The scoreboard - run this cell { display-mode: "form" }
import json
rows = list(table.items()) + [("CNN on SHUFFLED bound sites", auc_shuf_cnn),
                              ("Mistral-DNA on SHUFFLED bound sites", auc_shuf_fm)]
fig, ax = plt.subplots(figsize=(7, 3))
names, vals = zip(*rows)
colors = ["tab:blue", "tab:purple", "tab:grey", "tab:grey", "tab:blue", "tab:purple"]
ax.barh(names, vals, color=colors, alpha=.8)
ax.axvline(0.5, ls="--", c="k", lw=1); ax.set_xlim(0.45, 0.95); ax.invert_yaxis()
for i, v in enumerate(vals):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=8)
ax.set_xlabel("test AUROC"); plt.tight_layout(); plt.show()
_ = (DATA_DIR / "my_slot6_results.json").write_text(json.dumps(dict(rows), indent=1))

### Discuss

1. How much of each model's score is available from **two numbers**?
2. Which model barely noticed that its input was shuffled? What does that say it was reading?
3. Two numbers (G+C, U) already work. Is that the model's fault — or the **data's**? Where do the
   negatives come from, and why might they be made of different letters?

Slot 7: fix the data, not the model.